# Assamese OCR - Kaggle Training

**Setup:**
1. Enable GPU: Settings → Accelerator → GPU T4 x2
2. Add wiki dataset via "+ Add Input"
3. Update `WIKI_PATH` below to match your input path

---

## 1. Clone & Setup

In [ ]:
import os
import shutil

os.makedirs('/kaggle/working', exist_ok=True)
os.chdir('/kaggle/working')

REPO = 'AXOMIYA_OCR'
if os.path.exists(REPO):
    shutil.rmtree(REPO)

!git clone https://github.com/dikshitadutta/AXOMIYA_OCR.git
os.chdir(f'{REPO}/django_backend')
print(f'✅ Working in: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Setup Data

**To add your wiki file:**
1. Click "+ Add Input" button on the right
2. Click "Upload" tab
3. Upload `as-wiki-2021.txt`
4. It will appear at `/kaggle/input/as-wiki-2021/as-wiki-2021.txt`
5. Update the path below if different

In [ ]:
import os

os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# Update this if your upload path is different
WIKI_PATH = '/kaggle/input/as-wiki-2021/as-wiki-2021.txt'

if os.path.exists(WIKI_PATH):
    if not os.path.exists('data/as-wiki-2021.txt'):
        os.symlink(WIKI_PATH, 'data/as-wiki-2021.txt')
    print(f'✅ Wiki file ready ({os.path.getsize(WIKI_PATH) / 1e6:.1f} MB)')
else:
    print(f'❌ Wiki file not found at: {WIKI_PATH}')
    print('Please upload via "+ Add Input" → "Upload" button')

## 3. Generate Training Data

In [ ]:
# Check if splits already exist
train_labels = 'data/train_real_sentences/labels/labels.txt'
if os.path.exists(train_labels):
    with open(train_labels) as f:
        n = sum(1 for _ in f)
    print(f'✅ Splits already exist! ({n} training samples)')
    print('Skip the next cell unless you want to regenerate.')
else:
    print('❌ No splits found. Run the next cell to generate them.')

In [ ]:
# Step 1: Clean up any old split directories
for split in ['train_real_sentences', 'val_real_sentences', 'test_real_sentences']:
    dst = f'data/{split}'
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        shutil.rmtree(dst)
    os.makedirs(f'{dst}/images', exist_ok=True)
    os.makedirs(f'{dst}/labels', exist_ok=True)

print('✅ Local data directories ready')

# Step 2: Generate splits
!python build_real_sentence_splits.py \
    --input data/as-wiki-2021.txt \
    --train-output data/train_real_sentences \
    --val-output data/val_real_sentences \
    --test-output data/test_real_sentences \
    --train-count 50000 \
    --val-count 5000 \
    --test-count 2000 \
    --seed 42

# Step 3: Verify generated splits
print()
for split in ['train_real_sentences', 'val_real_sentences', 'test_real_sentences']:
    img_dir    = f'data/{split}/images'
    label_file = f'data/{split}/labels/labels.txt'
    n_images   = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    ok         = '✅' if os.path.exists(label_file) else '❌'
    print(f'{split}')
    print(f'  Images     : {n_images}')
    print(f'  labels.txt : {ok}')

print('\n✅ Dataset generation complete')

## 4. Train Model

**Auto-resume enabled** - saves checkpoint after every epoch

In [ ]:
# Check for existing checkpoint
latest = 'checkpoints/latest_model_sentences.pth'
if os.path.exists(latest):
    import torch
    ckpt = torch.load(latest, map_location='cpu')
    if isinstance(ckpt, dict) and 'epoch' in ckpt:
        print(f'✅ Resuming from epoch {ckpt["epoch"] + 1}')
else:
    print('Starting from scratch')

In [ ]:
!python train_sentence_model.py \
    --train-img-dir data/train_real_sentences/images \
    --train-label-file data/train_real_sentences/labels/labels.txt \
    --val-img-dir data/val_real_sentences/images \
    --val-label-file data/val_real_sentences/labels/labels.txt \
    --epochs 25 \
    --batch-size 32 \
    --learning-rate 0.0001 \
    --best-checkpoint checkpoints/best_model_sentences.pth \
    --final-checkpoint checkpoints/final_model_sentences.pth \
    --latest-checkpoint checkpoints/latest_model_sentences.pth \
    --resume \
    --plot-out training_curve.png

In [ ]:
from IPython.display import Image
if os.path.exists('training_curve.png'):
    display(Image('training_curve.png'))

## 5. Evaluate

In [ ]:
!python evaluate.py \
    --img-dir data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --num-samples 10

In [ ]:
!python evaluate.py \
    --img-dir data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --beam-width 10 \
    --num-samples 10

In [ ]:
!python evaluate.py \
    --img-dir data/test_real_sentences/images \
    --label-file data/test_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --num-samples 10 \
    --output test_results.tsv

## 6. Download Results

In [ ]:
print('Checkpoints:')
!ls -lh checkpoints/*.pth 2>/dev/null || echo 'None'
print('\nResults:')
!ls -lh *.png *.tsv 2>/dev/null || echo 'None'